In [4]:
#1. Data Processing
#Importing all the nesessary Libraries

# 1. Core Data Manipulation
import pandas as pd
from pathlib import Path
import warnings
warnings.simplefilter("ignore", UserWarning)

# 2. Pre-processing & Machine Learning
from sklearn.preprocessing import MinMaxScaler, StandardScaler # For scaling your data
from sklearn.impute import KNNImputer                          # For advanced missing data filling

# 3. Visualization (To verify your cleaning)
import matplotlib.pyplot as plt  # For plotting trends
import seaborn as sns            # For heatmaps and outlier boxplots

In [ ]:
#Data Consolidation Pipeline
#The code looks into your folder and finds all your CSV files and transforms individual data to clean unified database

In [8]:

import pandas as pd
from pathlib import Path


path = Path("Data1to28Patients")
# Combines your folder name and the *.csv pattern into a list of CSV files.
all_files = sorted(path.glob("*.csv"))

## li-- Initializes an empty list to hold the data from each file temporarily.
li = []

# This code transforms a messy folder of individual reports into a clean, unified database.
for filename in all_files:
    df = pd.read_csv(filename, sep=";", index_col=None, header=0)
    df['Patient_ID'] = filename.stem
    li.append(df)
    
#Merging everything
combined_activity = pd.concat(li, axis=0, ignore_index=True)
combined_activity.to_csv("Combined_Datasets_1to28.csv", index=False)

In [5]:
df.head(6)

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID
0,2020-01-17T00:00:00,40.000000,15.0429,96.371901,8.0,0.035,0.0,1.0,HUPA0023P
1,2020-01-17T00:05:00,41.333333,8.3164,91.395349,0.0,0.035,0.0,0.0,HUPA0023P
2,2020-01-17T00:10:00,42.666667,7.5826,85.991935,0.0,0.035,0.0,0.0,HUPA0023P
3,2020-01-17T00:15:00,44.000000,7.3380,82.434426,0.0,0.035,0.0,0.0,HUPA0023P
4,2020-01-17T00:20:00,50.000000,7.5826,78.822581,0.0,0.035,0.0,0.0,HUPA0023P
5,2020-01-17T00:25:00,56.000000,7.5826,81.101562,0.0,0.035,0.0,0.0,HUPA0023P


In [6]:
df.shape

(25902, 9)

In [5]:
#Change time data type to datetime

In [22]:
df['time'] = pd.to_datetime(df['time'])
df

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID,hour
0,2020-01-17 00:00:00,40.000,15.043,96.372,8.0,0.035,0.0,1.0,HUPA0023P,0
1,2020-01-17 00:05:00,41.333,8.316,91.395,0.0,0.035,0.0,0.0,HUPA0023P,0
2,2020-01-17 00:10:00,42.667,7.583,85.992,0.0,0.035,0.0,0.0,HUPA0023P,0
3,2020-01-17 00:15:00,44.000,7.338,82.434,0.0,0.035,0.0,0.0,HUPA0023P,0
4,2020-01-17 00:20:00,50.000,7.583,78.823,0.0,0.035,0.0,0.0,HUPA0023P,0
...,...,...,...,...,...,...,...,...,...,...
3914,2020-01-30 14:10:00,76.000,6.727,67.932,0.0,0.035,0.0,0.0,HUPA0023P,14
3915,2020-01-30 14:15:00,76.000,12.841,69.116,39.0,0.035,0.0,0.0,HUPA0023P,14
3916,2020-01-30 14:20:00,76.333,6.604,63.786,0.0,0.035,0.0,0.0,HUPA0023P,14
3917,2020-01-30 14:25:00,76.667,6.849,69.115,0.0,0.035,0.0,0.0,HUPA0023P,14


In [ ]:
#Rounding the columns to 3 decimals

In [10]:

cols_to_round = ["glucose", "calories", "heart_rate"]

df[cols_to_round] = df[cols_to_round].round(3)


In [14]:
df.dtypes

time                       object
glucose                   float64
calories                  float64
heart_rate                float64
steps                     float64
basal_rate                float64
bolus_volume_delivered    float64
carb_input                float64
Patient_ID                 object
dtype: object

In [ ]:
#Standardizing Time zones If some data was recorded in UTC and some in EST, 
#it strips those labels so they can be compared easily as "naive" local
#time and sort time zones in chronlogical order removing duplicates

In [21]:
# Convert to datetime and strip timezone info
df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)

# Sort and remove duplicates (keep the first occurrence)
df = df.sort_values(['Patient_ID', 'time'])
df = df.drop_duplicates(subset=['Patient_ID', 'time'], keep='first').reset_index(drop=True)

In [15]:
df.head()

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID
0,2020-01-17T00:00:00,40.000000,15.0429,96.371901,8.0,0.035,0.0,1.0,HUPA0023P
1,2020-01-17T00:05:00,41.333333,8.3164,91.395349,0.0,0.035,0.0,0.0,HUPA0023P
2,2020-01-17T00:10:00,42.666667,7.5826,85.991935,0.0,0.035,0.0,0.0,HUPA0023P
3,2020-01-17T00:15:00,44.000000,7.3380,82.434426,0.0,0.035,0.0,0.0,HUPA0023P
4,2020-01-17T00:20:00,50.000000,7.5826,78.822581,0.0,0.035,0.0,0.0,HUPA0023P


In [ ]:
#Checking for Outliers

In [9]:
numeric_cols = [
    'glucose',
    'calories',
    'heart_rate',
    'steps',
    'basal_rate',
    'bolus_volume_delivered',
    'carb_input'
]

# Compute IQR
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

# Outlier mask (DataFrame of True/False)
outliers = (df[numeric_cols] < (Q1 - 1.5 * IQR)) | \
           (df[numeric_cols] > (Q3 + 1.5 * IQR))

# Count outliers per column
outliers.sum()

glucose                    255
calories                  3910
heart_rate                 852
steps                     5526
basal_rate                   0
bolus_volume_delivered     223
carb_input                 217
dtype: int64

In [17]:
df.head(3)

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID
0,2020-01-17T00:00:00,40.000000,15.0429,96.371901,8.0,0.035,0.0,1.0,HUPA0023P
1,2020-01-17T00:05:00,41.333333,8.3164,91.395349,0.0,0.035,0.0,0.0,HUPA0023P
2,2020-01-17T00:10:00,42.666667,7.5826,85.991935,0.0,0.035,0.0,0.0,HUPA0023P


In [18]:
df.tail(3)


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID
25899,2022-05-18 12:05:00,118.666667,5.66622,95.542857,0.0,0.0,0.0,0.0,HUPA0028P
25900,2022-05-18 12:10:00,123.333333,5.57628,91.381356,0.0,0.0,0.0,0.0,HUPA0028P
25901,2022-05-18 12:15:00,128.000000,5.57628,99.257812,0.0,0.0,0.0,0.0,HUPA0028P


In [13]:
# 1. Convert to datetime and strip timezone info
df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)

# 2. Sort using df_cleaned
df = df.sort_values(['Patient_ID','time'])

# 3. Drop duplicates using df_cleaned
df = df.drop_duplicates(subset=['Patient_ID','time'], keep='first').reset_index(drop=True)

In [21]:
df.shape

(25902, 9)

In [15]:
print(f"Count before cleaning: {df.shape[0]}")
print(f"Count after cleaning:  {df.shape[0]}")

Count before cleaning: 25902
Count after cleaning:  25902


In [20]:
df.head()

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID
0,2022-02-17 13:50:00,78.333333,13.58094,86.040323,198.0,0.0,0.0,0.0,HUPA0028P
1,2022-02-17 13:55:00,77.666667,28.42104,121.808511,374.0,0.0,0.0,0.0,HUPA0028P
2,2022-02-17 14:00:00,77.000000,15.91938,101.689394,96.0,0.0,0.0,0.0,HUPA0028P
3,2022-02-17 14:05:00,76.666667,8.81412,91.308725,0.0,0.0,0.0,0.0,HUPA0028P
4,2022-02-17 14:10:00,76.333333,8.45436,93.473684,0.0,0.0,0.0,0.0,HUPA0028P


In [6]:
import pandas as pd
from pathlib import Path


path = Path("Data1to28Patients")
# Combines your folder name and the *.csv pattern into a list of CSV files.
all_files = sorted(path.glob("*.csv"))

## li-- Initializes an empty list to hold the data from each file temporarily.
li = []

# This code transforms a messy folder of individual reports into a clean, unified database.
for filename in all_files:
    df = pd.read_csv(filename, sep=";", index_col=None, header=0)
    df['Patient_ID'] = filename.stem
    li.append(df)
    
#Merging everything
combined_activity = pd.concat(li, axis=0, ignore_index=True)
combined_activity.to_csv("Combined_Datasets_1to28.csv", index=False)
df=combined_activity #merge

In [7]:
df

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID
0,2018-06-13T18:40:00,332.000000,6.35950,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P
1,2018-06-13T18:45:00,326.000000,7.72800,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P
2,2018-06-13T18:50:00,330.000000,4.74950,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P
3,2018-06-13T18:55:00,324.000000,6.35950,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P
4,2018-06-13T19:00:00,306.000000,5.15200,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P
...,...,...,...,...,...,...,...,...,...
309387,2022-05-18T11:55:00,109.333333,10.79280,104.171171,0.0,0.000000,0.0,0.0,HUPA0028P
309388,2022-05-18T12:00:00,114.000000,9.80346,103.442623,0.0,0.000000,0.0,0.0,HUPA0028P
309389,2022-05-18T12:05:00,118.666667,5.66622,95.542857,0.0,0.000000,0.0,0.0,HUPA0028P
309390,2022-05-18T12:10:00,123.333333,5.57628,91.381356,0.0,0.000000,0.0,0.0,HUPA0028P
